# SCI China Oil & Chemical — Data Audit

This notebook traces **where data comes from**, **how it is transformed**, and **what the cached tables look like**.

**Pipeline summary**

1. **SCI99 API** (`open.sci99.com/datapackapi`) — package **1233** (*China Oil & Chemical Industry Data - Ownership*)
2. **`fetch_data.py --history`** — pulls 3 snapshots (2024-12-31, 2025-12-31, latest), merges time series, writes parquet
3. **`app.py`** — reads parquet from `data/cache/` (no live API on tab click)

Set `CALL_API = True` below to hit the API from this notebook; otherwise it uses the local cache only.

In [17]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Project root = parent of notebooks/
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

load_dotenv(ROOT / ".env")

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

CALL_API = True  # flip to True to authenticate and fetch live
REFRESH_CACHE = False  # if True with CALL_API, re-run fetch_and_cache

print("Project root:", ROOT)
print("USER_NAME set:", bool(os.getenv("USER_NAME")))
print("DATA_PACKAGE_ID:", os.getenv("DATA_PACKAGE_ID", "1233 (default)"))

Project root: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api
USER_NAME set: True
DATA_PACKAGE_ID: 1233


## 1. API endpoints & authentication

| Step | Method | URL |
|------|--------|-----|
| Token | POST | `https://open.sci99.com/datapackapi/pkg/user/gettoken` |
| Package list | GET | `https://open.sci99.com/datapackapi/pkg/getmypkglist` |
| Package data | GET | `https://open.sci99.com/datapackapi/pkg/getpkgdata?pkgid=…&pubdate=…` |

Credentials live in `sci_api/.env` (`USER_NAME`, `PASSWORD`, optional `DATA_PACKAGE_ID`).

In [18]:
from src.client import SciClient

client = SciClient()

if CALL_API:
    token = client.authenticate()
    print("Token obtained:", bool(token))
    packages = client.list_packages()
    display(pd.DataFrame(packages)[["DataPackageID", "DataPackageName", "LatestDate"]])
    latest_pubdate = client.latest_pubdate()
    print("Latest pubdate for pkg", client.package_id, ":", latest_pubdate)
else:
    print("Skipping API (CALL_API=False). Enable CALL_API to inspect live responses.")

Token obtained: True


,DataPackageID,DataPackageName,LatestDate
0,742,SCI China Natural Gas Forecast Datapack,2026-07-03T00:00:00
1,782,SCI China Natural Gas Datapack-Key Drivers,2026-07-03T00:00:00
2,1233,SCI China Oil & Chemical Industry Data - Owner...,2026-07-06T00:00:00


Latest pubdate for pkg 1233 : 2026-07-06T00:00:00


## 2. Raw API payload shape

Each package returns JSON like:

```json
{
  "DataPackageName": "…",
  "PubDate": "2026/6/26 0:00:00",
  "SheetList": [
    {
      "SheetName": "03.Weekly summary",
      "Topic": "…",
      "Unit": "kt",
      "DataList": [ { … row dict … }, … ]
    }
  ]
}
```

Each row in `DataList` is a **flat dict** (API is cleaner than the Excel wide layout).

In [19]:
if CALL_API:
    pkg = client.fetch_package()
    print("Package:", pkg["DataPackageName"])
    print("PubDate:", pkg["PubDate"])
    print("Sheets:", len(pkg["SheetList"]))

    sheet_catalog = pd.DataFrame([
        {
            "SheetName": s["SheetName"],
            "Topic": (s.get("Topic") or "")[:60],
            "Unit": s.get("Unit"),
            "Rows": len(s.get("DataList", [])),
        }
        for s in pkg["SheetList"]
    ])
    display(sheet_catalog.sort_values("SheetName"))

    # Example: first row of weekly summary
    ws = client.sheet_by_name(pkg, "weekly", "summary")
    print("\nWeekly summary — first record keys:")
    print(list(ws["DataList"][0].keys())[:12], "…")
    display(pd.DataFrame(ws["DataList"]).head(3))
else:
    print("Enable CALL_API to inspect raw SheetList.")

Package: SCI China Oil & Chemical Industry Data - Ownership
PubDate: 2026/7/6 0:00:00
Sheets: 48


,SheetName,Topic,Unit,Rows
0,01.Balance,China Oil & Chemical Industry Supply-Demand Ba...,,45
1,02.Monthly summary,2026 China Oil & Chemical Industry Monthly Sum...,,123
2,03.Weekly summary,China Oil & Chemical Industry Weekly Summary (...,,81
3,04.Capacity 1,2026 China Major Oil Refining Unit Capacity St...,,1950
4,05.Capacity 2,China Oil Refining Unit Capacity (kt/a),,327
5,06.Capacity 2 revision,Data Revise Record (kt/a or Nm3/h),,19
6,07.Processing volume,2026 China Unit Feedstock Processing Volume (b...,,899
7,08.Processing volume - Ownership,2026 China Unit Feedstock Processing Volume (b...,,542
9,10.Operating rate - Ownership,2026 China Oil & Chemical Unit Operating Rate ...,,542
10,11.Oil product output,2026 Oil Refining Product Output by Province (kt),,1426



Weekly summary — first record keys:
['Breakdown', 'Item', 'Unit', '1st week', '2nd week', '3rd week', '4th week', '5th week', '6th week', '7th week', '8th week', '9th week'] …


,Breakdown,Item,Unit,1st week,2nd week,3rd week,4th week,5th week,6th week,7th week,8th week,9th week,10th week,11th week,12th week,...,43rd week,44th week,45th week,46th week,47th week,48th week,49th week,50th week,51st week,52nd week,53rd week,54th week,55th week,56th week,57th week
0,Breakdown,Item,Unit,2026-01-02,2026-01-09,2026-01-16,2026-01-23,2026-01-30,2026-02-06,2026-02-13,2026-02-27,2026-03-06,2026-03-13,2026-03-20,2026-03-27,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
1,National,Feedstock primary processing volume (kt),kt,14119.2,14127.0,14150.6,14355.2,14424.0,14685.3,14734.2,14732.3,14660.2,14189.8,13738.7,13209.2,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None
2,National,Feedstock primary processing unit operating rate,-,0.7776,0.778,0.7793,0.7906,0.7944,0.8101,0.8128,0.8127,0.8087,0.7828,0.7579,0.7287,...,None,None,None,None,None,None,None,None,None,None,None,None,None,None,None


## 3. Historical merge (`fetch_merged_history`)

Defined in `src/history.py`. Fetches **three pubdates**:

- `2024-12-31` — full 2024 weekly year + 2024 monthly consumption/refueling
- `2025-12-31` — full 2025
- **Latest** (e.g. `2026-06-26`) — current-year YTD

Weekly / stocks / margins / consumption rows are **deduplicated** on time keys (latest snapshot wins on overlap).

In [20]:
from src.history import HISTORY_PUBDATES, fetch_merged_history
from fetch_data import CACHE_DIR, FRAME_NAMES, load_cached_frames, fetch_and_cache

print("History snapshots:", HISTORY_PUBDATES)

if REFRESH_CACHE and CALL_API:
    cache_path = fetch_and_cache(include_history=True)
    print("Wrote cache:", cache_path)

frames = load_cached_frames()
if frames is None and CALL_API:
    frames = fetch_merged_history(include_history=True)
elif frames is None:
    raise FileNotFoundError(
        "No cache found. Run: python fetch_data.py --history\n"
        "Or set CALL_API=True and REFRESH_CACHE=True in this notebook."
    )

print("Loaded frames:", sorted(frames.keys()))
display(frames["meta"])

History snapshots: ('2021-12-31T00:00:00', '2022-12-31T00:00:00', '2023-12-31T00:00:00', '2024-12-31T00:00:00', '2025-12-31T00:00:00')
Loaded frames: ['balance_monthly', 'consumption', 'consumption_national', 'export_forecast', 'export_quotas_monthly', 'export_quotas_summary', 'export_shipping', 'kerosene_refueling', 'kerosene_refueling_national', 'margins', 'margins_wide', 'meta', 'operating_rate_ownership', 'petchem_inventories', 'petchem_monthly', 'petchem_weekly', 'processing_ownership', 'products_stocks_wide', 'provincial_output', 'provincial_processing', 'runs_wide', 'shandong_stocks', 'shandong_stocks_total', 'weekly']


,package_name,pub_date,fetched_at,history_snapshots
0,SCI China Oil & Chemical Industry Data - Owner...,2026-07-06,2026-07-08 10:32:02.164238+00:00,"['2021-12-31T00:00:00', '2022-12-31T00:00:00',..."


## 4. Cached parquet inventory

The dashboard reads from `data/cache/<pubdate>/` (pointer in `latest.txt` on Windows).

In [21]:
latest_txt = CACHE_DIR / "latest.txt"
if latest_txt.exists():
    cache_dir = CACHE_DIR / latest_txt.read_text(encoding="utf-8").strip()
else:
    dirs = sorted(p for p in CACHE_DIR.iterdir() if p.is_dir() and p.name.isdigit())
    cache_dir = dirs[-1] if dirs else CACHE_DIR

print("Cache dir:", cache_dir)
for name in FRAME_NAMES:
    p = cache_dir / f"{name}.parquet"
    if p.exists():
        df = pd.read_parquet(p)
        print(f"{name:30} {str(df.shape):>12}  {p.stat().st_size / 1024:,.0f} KB")
    else:
        print(f"{name:30} MISSING")

Cache dir: C:\Users\luiscarlos.gaitan\OneDrive - Jain Global\Coding\sci_api\data\cache\20260706
meta                                 (1, 4)  4 KB
weekly                            (3894, 5)  34 KB
runs_wide                          (825, 4)  16 KB
products_stocks_wide               (275, 6)  12 KB
shandong_stocks                  (10248, 4)  29 KB
shandong_stocks_total             (1104, 3)  12 KB
processing_ownership              (3252, 5)  15 KB
operating_rate_ownership          (3252, 5)  16 KB
balance_monthly                     (65, 5)  6 KB
consumption                      (12276, 6)  55 KB
consumption_national               (396, 4)  5 KB
kerosene_refueling                (4092, 5)  16 KB
kerosene_refueling_national        (198, 4)  4 KB
margins                          (25168, 6)  159 KB
margins_wide                      (176, 16)  39 KB
provincial_processing             (2046, 5)  11 KB
provincial_output                (11904, 6)  50 KB
petchem_weekly                    (2727,

## 5. Normalized tables — schema & date coverage

All dashboard charts use these **tidy** parquet tables (not raw API JSON).

In [22]:
def audit_frame(name: str, df: pd.DataFrame, date_col: str | None = None) -> None:
    print(f"\n{'='*60}\n{name}  shape={df.shape}")
    display(df.dtypes.to_frame("dtype"))
    if date_col and date_col in df.columns:
        d = pd.to_datetime(df[date_col], errors="coerce")
        print(f"  {date_col}: {d.min()} → {d.max()}  ({d.notna().sum()} rows)")
    display(df.head(3))

audit_frame("weekly", frames["weekly"], "week_start")
audit_frame("runs_wide", frames["runs_wide"], "week_start")
audit_frame("products_stocks_wide", frames["products_stocks_wide"], "week_start")
audit_frame("balance_monthly", frames["balance_monthly"], "month_start")


weekly  shape=(3894, 5)


,dtype
week_start,datetime64[ns]
breakdown,object
metric,object
value,float64
unit,object


  week_start: 2021-01-08 00:00:00 → 2026-07-03 00:00:00  (3894 rows)


,week_start,breakdown,metric,value,unit
0,2021-01-08,Independents,diesel_output_kt,2146.55,kt
1,2021-01-08,National,gasoline_output_kt,3722.11,kt
2,2021-01-08,SOEs,diesel_output_kt,2449.50,kt



runs_wide  shape=(825, 4)


...

,week_start,diesel_output_kt,diesel_stocks_kt,gasoline_output_kt,gasoline_stocks_kt,kerosene_output_kt
0,2021-01-08,4596.05,NaN,3722.11,NaN,NaN
1,2021-01-15,4590.85,NaN,3717.45,NaN,NaN
2,2021-01-22,4588.55,NaN,3714.55,NaN,NaN



balance_monthly  shape=(65, 5)


,dtype
month_start,datetime64[ns]
crude_processing_kt,float64
cdu_operating_rate,float64
gasoline_stocks_kt,float64
diesel_stocks_kt,float64


  month_start: 2021-01-01 00:00:00 → 2026-05-01 00:00:00  (65 rows)


,month_start,crude_processing_kt,cdu_operating_rate,gasoline_stocks_kt,diesel_stocks_kt
0,2021-01-01,59792.3,0.7362,25071.4,26037.7
1,2021-02-01,54766.0,0.7466,26273.7,27413.3
2,2021-03-01,58749.0,0.7232,27120.2,27296.2


## 6. Consumption tab — source sheets & transforms

| Dashboard panel | API sheet | Parser | Aggregation |
|-----------------|-----------|--------|-------------|
| Gasoline, Diesel, … | `23.Consumption` | `parse_consumption()` | Sum provinces → national |
| Jet fuel (intl/dom/total) | `25.Kerosene refueling volume` | `parse_kerosene_refueling()` | Sum provinces; total = intl + domestic |

Raw API layout: wide monthly columns (`Jan`…`Dec`) per **province × product** (consumption) or **province × flight type** (refueling).  
Calendar **year** is taken from the snapshot `PubDate` (each file holds one calendar year).

In [23]:
audit_frame("consumption (provincial long)", frames["consumption"])
print("Products:", sorted(frames["consumption"]["product"].unique()))
print("Years:", sorted(frames["consumption"]["year"].unique()))

audit_frame("consumption_national", frames["consumption_national"])
display(
    frames["consumption_national"]
    .groupby(["product", "year"])["month"]
    .count()
    .unstack(fill_value=0)
)


consumption (provincial long)  shape=(12276, 6)


,dtype
product,object
region,object
province,object
year,int64
month,int64
consumption_kt,float64


,product,region,province,year,month,consumption_kt
0,Diesel,E-China,Anhui,2021,1,551.7
1,Diesel,E-China,Anhui,2021,2,500.4
2,Diesel,E-China,Anhui,2021,3,527.0


Products: ['Diesel', 'Gasoline', 'Kerosene', 'Naphtha', 'Residue', 'VGO']
Years: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025), np.int64(2026)]

consumption_national  shape=(396, 4)


,dtype
product,object
year,int64
month,int64
consumption_kt,float64


,product,year,month,consumption_kt
0,Diesel,2021,1,16616.7
1,Diesel,2021,2,14065.7
2,Diesel,2021,3,15636.4


year,2021,2022,2023,2024,2025,2026
product,,,,,,
Diesel,12,12,12,12,12,6
Gasoline,12,12,12,12,12,6
Kerosene,12,12,12,12,12,6
Naphtha,12,12,12,12,12,6
Residue,12,12,12,12,12,6
VGO,12,12,12,12,12,6


In [24]:
audit_frame("kerosene_refueling (provincial long)", frames["kerosene_refueling"])
audit_frame("kerosene_refueling_national", frames["kerosene_refueling_national"])

# National jet fuel total — recent months
jet = frames["kerosene_refueling_national"]
display(
    jet[jet["product"] == "Jet fuel (total)"]
    .sort_values(["year", "month"], ascending=[False, False])
    .head(12)
)


kerosene_refueling (provincial long)  shape=(4092, 5)


,dtype
product,object
province,object
year,int64
month,int64
consumption_kt,float64


,product,province,year,month,consumption_kt
0,Jet fuel (domestic),Anhui,2021,1,28.6
1,Jet fuel (domestic),Anhui,2021,2,22.9
2,Jet fuel (domestic),Anhui,2021,3,37.7



kerosene_refueling_national  shape=(198, 4)


,dtype
year,int64
month,int64
consumption_kt,float64
product,object


,year,month,consumption_kt,product
0,2021,1,2262.8,Jet fuel (domestic)
1,2021,2,1737.9,Jet fuel (domestic)
2,2021,3,2950.4,Jet fuel (domestic)


,year,month,consumption_kt,product
197,2026,6,3908.8,Jet fuel (total)
196,2026,5,3990.8,Jet fuel (total)
195,2026,4,4105.1,Jet fuel (total)
194,2026,3,4433.2,Jet fuel (total)
193,2026,2,4433.2,Jet fuel (total)
192,2026,1,4327.5,Jet fuel (total)
191,2025,12,4166.7,Jet fuel (total)
190,2025,11,4060.3,Jet fuel (total)
189,2025,10,4449.4,Jet fuel (total)
188,2025,9,4197.1,Jet fuel (total)


## 7. Raw API vs parsed — side-by-side (consumption)

Run with `CALL_API=True` to compare one snapshot before/after `parse_consumption`.

In [25]:
from src.transforms import parse_consumption, parse_kerosene_refueling

if CALL_API:
    pubdate = "2026-06-26T00:00:00"
    pkg = client.fetch_package(pubdate=pubdate)
    pub_year = int(pd.to_datetime(pkg["PubDate"]).year)

    cons_sheet = client.sheet_by_name(pkg, "consumption")
    raw_cons = pd.DataFrame(cons_sheet["DataList"])
    print("RAW consumption sheet (wide):")
    display(raw_cons.head(4))

    parsed_cons = parse_consumption(cons_sheet, calendar_year=pub_year)
    print(f"\nPARSED (long) — calendar_year={pub_year}:")
    display(parsed_cons.head(8))

    ref_sheet = client.sheet_by_name(pkg, "kerosene", "refueling")
    raw_ref = pd.DataFrame(ref_sheet["DataList"])
    print("\nRAW kerosene refueling:")
    display(raw_ref.head(4))

    parsed_ref = parse_kerosene_refueling(ref_sheet, calendar_year=pub_year)
    print("\nPARSED refueling:")
    display(parsed_ref.head(8))
else:
    print("Set CALL_API=True to compare raw vs parsed.")

RAW consumption sheet (wide):


,Product,Region,Province,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,Gasoline,E-China,Anhui,461.8,519.8,475.5,367.2,379.5,None,None,None,None,None,None,None
1,Gasoline,N-China,Beijing,268.2,228.5,224.6,209.7,227.8,None,None,None,None,None,None,None
2,Gasoline,E-China,Fujian,433.2,416.7,441.1,353.9,373.5,None,None,None,None,None,None,None
3,Gasoline,NW-China,Gansu,178.6,159.4,149.3,110.7,126.2,None,None,None,None,None,None,None



PARSED (long) — calendar_year=2026:


,product,region,province,year,month,consumption_kt
0,Diesel,E-China,Anhui,2026,1,486.4
1,Diesel,N-China,Beijing,2026,1,110.7
2,Diesel,SW-China,Chongqing,2026,1,416.8
3,Diesel,E-China,Fujian,2026,1,577.5
4,Diesel,NW-China,Gansu,2026,1,256.4
5,Diesel,S-China,Guangdong,2026,1,1861.3
6,Diesel,S-China,Guangxi,2026,1,588.5
7,Diesel,SW-China,Guizhou,2026,1,440.9



RAW kerosene refueling:


,Item,Province,Jan,Feb,Mar,Apr,May,Jun,Jul,Aug,Sep,Oct,Nov,Dec
0,International flight,Anhui,1.8,2.1,1.8,2.1,2.0,None,None,None,None,None,None,None
1,International flight,Beijing,130.8,116.8,123.7,124.5,124.7,None,None,None,None,None,None,None
2,International flight,Fujian,41.8,40.7,41.8,40.1,38.2,None,None,None,None,None,None,None
3,International flight,Gansu,0.5,0.3,0.2,0.1,0.3,None,None,None,None,None,None,None



PARSED refueling:


,product,province,year,month,consumption_kt
0,Jet fuel (domestic),Anhui,2026,1,43.1
1,Jet fuel (domestic),Beijing,2026,1,245.0
2,Jet fuel (domestic),Chongqing,2026,1,119.7
3,Jet fuel (domestic),Fujian,2026,1,120.1
4,Jet fuel (domestic),Gansu,2026,1,50.0
5,Jet fuel (domestic),Guangdong,2026,1,397.8
6,Jet fuel (domestic),Guangxi,2026,1,58.9
7,Jet fuel (domestic),Guizhou,2026,1,63.6


## 8. Weekly runs & unit conversion

- **Weekly summary** sheet → `weekly` (long) → `runs_wide` / `products_stocks_wide`
- **kb/d** display: `weekly_kt / 7 × 7.33` (see `src/units.py`)
- **Monthly consumption kb/d**: `monthly_kt / days_in_month × 7.33`

In [26]:
from src.units import weekly_kt_to_kbd, monthly_kt_to_kbd

runs = frames["runs_wide"]
latest = runs[runs["breakdown"] == "National"].sort_values("week_start").tail(1)
display(latest)
if not latest.empty:
    kt = latest["processing_kt"].iloc[0]
    print(f"Latest national weekly processing: {kt:,.1f} kt/week ≈ {weekly_kt_to_kbd(kt):,.0f} kb/d")

gas = frames["consumption_national"]
row = gas[(gas["product"] == "Gasoline") & (gas["year"] == gas["year"].max())].sort_values("month").tail(1)
if not row.empty:
    ms = pd.Timestamp(year=int(row["year"].iloc[0]), month=int(row["month"].iloc[0]), day=1)
    kt_m = row["consumption_kt"].iloc[0]
    print(f"Latest gasoline consumption month: {kt_m:,.1f} kt/month ≈ {monthly_kt_to_kbd(kt_m, pd.Series([ms])).values[0]:,.0f} kb/d avg")

,week_start,breakdown,operating_rate,processing_kt
823,2026-07-03,National,0.6358,11525.39


Latest national weekly processing: 11,525.4 kt/week ≈ 12,069 kb/d
Latest gasoline consumption month: 9,922.7 kt/month ≈ 2,424 kb/d avg


## 9. Margins (Economics tab)

From sheet `18.Profit model` — wide weekly columns melted to long, then `margins_wide` pivots profit metrics.

In [27]:
audit_frame("margins (long)", frames["margins"], "week_start")
print("\nProfit margin columns in margins_wide:")
margin_cols = [c for c in frames["margins_wide"].columns if c != "week_start"]
print(margin_cols)
display(frames["margins_wide"].tail(3))


margins (long)  shape=(25168, 6)


,dtype
week_start,datetime64[ns]
refinery_type,object
section,object
factor,object
unit,object
value,float64


  week_start: 2023-01-06 00:00:00 → 2026-07-03 00:00:00  (25168 rows)


,week_start,refinery_type,section,factor,unit,value
0,2023-01-06,Fuel-oriented refineries,Cost,Brent,$/bbl,78.69
1,2023-01-06,Oil Refining and Chemical Complexes,Product price,Benzene,RMB/mt,6710.00
2,2023-01-06,Oil Refining and Chemical Complexes,Product price,Butadiene,RMB/mt,8050.00



Profit margin columns in margins_wide:
['Fueloriented_refineries_contribution_margin', 'Fueloriented_refineries_gross_profit_margin_pct', 'Fueloriented_refineries_net_profit', 'Fueloriented_refineries_net_profit_margin_pct', 'Fueloriented_refineries_profit_per_ton', 'Oil_Refining_and_Aromatics_Complexes_contribution_margin', 'Oil_Refining_and_Aromatics_Complexes_gross_profit_margin_pct', 'Oil_Refining_and_Aromatics_Complexes_net_profit', 'Oil_Refining_and_Aromatics_Complexes_net_profit_margin_pct', 'Oil_Refining_and_Aromatics_Complexes_profit_per_ton', 'Oil_Refining_and_Chemical_Complexes_contribution_margin', 'Oil_Refining_and_Chemical_Complexes_gross_profit_margin_pct', 'Oil_Refining_and_Chemical_Complexes_net_profit', 'Oil_Refining_and_Chemical_Complexes_net_profit_margin_pct', 'Oil_Refining_and_Chemical_Complexes_profit_per_ton']


,week_start,Fueloriented_refineries_contribution_margin,Fueloriented_refineries_gross_profit_margin_pct,Fueloriented_refineries_net_profit,Fueloriented_refineries_net_profit_margin_pct,Fueloriented_refineries_profit_per_ton,Oil_Refining_and_Aromatics_Complexes_contribution_margin,Oil_Refining_and_Aromatics_Complexes_gross_profit_margin_pct,Oil_Refining_and_Aromatics_Complexes_net_profit,Oil_Refining_and_Aromatics_Complexes_net_profit_margin_pct,Oil_Refining_and_Aromatics_Complexes_profit_per_ton,Oil_Refining_and_Chemical_Complexes_contribution_margin,Oil_Refining_and_Chemical_Complexes_gross_profit_margin_pct,Oil_Refining_and_Chemical_Complexes_net_profit,Oil_Refining_and_Chemical_Complexes_net_profit_margin_pct,Oil_Refining_and_Chemical_Complexes_profit_per_ton
173,2026-06-18,26770173.20,0.3397,-451293.94,-0.0057,-34.71,86138281.49,0.4056,33322767.33,0.1569,1056.42,2.162137e+08,0.3005,71621247.93,0.0996,569.72
174,2026-06-26,26916025.58,0.3531,-277912.86,-0.0036,-21.38,86881576.85,0.4207,34242259.00,0.1658,1085.57,2.130546e+08,0.3088,69411746.67,0.1006,552.14
175,2026-07-03,27958666.65,0.3732,769774.81,0.0103,59.21,81043553.22,0.4156,28818006.25,0.1478,913.61,2.074295e+08,0.3136,64722938.43,0.0978,514.84


In [28]:
frames['margins']

,week_start,refinery_type,section,factor,unit,value
0,2023-01-06,Fuel-oriented refineries,Cost,Brent,$/bbl,7.869000e+01
1,2023-01-06,Oil Refining and Chemical Complexes,Product price,Benzene,RMB/mt,6.710000e+03
2,2023-01-06,Oil Refining and Chemical Complexes,Product price,Butadiene,RMB/mt,8.050000e+03
3,2023-01-06,Oil Refining and Chemical Complexes,Product price,Diesel,RMB/mt,7.392610e+03
4,2023-01-06,Oil Refining and Chemical Complexes,Product price,EVA,RMB/mt,1.413333e+04
...,...,...,...,...,...,...
25163,2026-07-03,Oil Refining and Aromatics Complexes,Product price,LPG,RMB/mt,5.077714e+03
25164,2026-07-03,Oil Refining and Aromatics Complexes,Product price,PP,RMB/mt,7.900000e+03
25165,2026-07-03,Oil Refining and Aromatics Complexes,Product price,PX,RMB/mt,7.541758e+03
25166,2026-07-03,Oil Refining and Aromatics Complexes,Sales revenue,Slurry oil (taxes excluded),RMB/mt,4.039823e+03


## 10. Product export quotas (Exports tab)

| Dashboard panel | API sheet | Parser | Wide table |
|-----------------|-----------|--------|------------|
| Monthly quota utilization | `45.Product quotas` | `parse_export_quotas_monthly()` | `build_export_monthly_from_quotas()` |
| Quota batches & ceiling | same sheet | `parse_export_quotas_summary()` | — |

**Product mapping** (SCI API → dashboard columns):

| SCI `Product` | Column |
|---------------|--------|
| Gasoline | Gasoline |
| Kerosene | Jet |
| Diesel | Gasoil/Diesel |
| — | FO (always 0; not on quota sheet) |

Units: **kt**. Month labels: **`YYYY-MM`**. Sheet 45 from each history snapshot is merged in `fetch_merged_history`.

If `export_quotas_monthly` is missing from cache, run `python fetch_data.py --history` or set `REFRESH_CACHE=True`.

In [29]:
from src.transforms import (
    EXPORT_CARGO_LABELS,
    EXPORT_TABLE_PRODUCTS,
    build_export_monthly_from_quotas,
    parse_export_quotas_monthly,
    parse_export_quotas_summary,
)

quotas_monthly = frames.get("export_quotas_monthly", pd.DataFrame())
quotas_summary = frames.get("export_quotas_summary", pd.DataFrame())

if quotas_monthly.empty:
    print(
        "export_quotas_monthly not in cache — run: python fetch_data.py --history\n"
        "Or set REFRESH_CACHE=True with CALL_API."
    )
else:
    audit_frame("export_quotas_monthly (long)", quotas_monthly, "month_start")
    print("\nProduct labels after mapping:", dict(EXPORT_CARGO_LABELS))
    print("Wide columns:", EXPORT_TABLE_PRODUCTS)

if not quotas_summary.empty:
    print("\nAnnual quota summary (latest snapshot):")
    display(quotas_summary)


export_quotas_monthly (long)  shape=(36, 5)


,dtype
product,object
year,int32
month,int32
month_start,datetime64[ns]
export_kt,float64


  month_start: 2026-01-01 00:00:00 → 2026-12-01 00:00:00  (36 rows)


,product,year,month,month_start,export_kt
0,Gasoil/Diesel,2026,1,2026-01-01,416.0
1,Gasoil/Diesel,2026,2,2026-02-01,439.0
2,Gasoil/Diesel,2026,3,2026-03-01,770.0



Product labels after mapping: {'Gasoline': 'Gasoline', 'Kerosene': 'Jet', 'Diesel': 'Gasoil/Diesel'}
Wide columns: ['Gasoline', 'Jet', 'Gasoil/Diesel', 'FO']

Annual quota summary (latest snapshot):


,quota_year,batch_1_kt,batch_2_kt,batch_3_kt,total_quota_kt,ytd_export_kt,utilization
0,2026,19000.0,13000.0,None,32000.0,16055.0,0.501719


In [38]:
frames['export_quotas_monthly']

,product,year,month,month_start,export_kt
0,Gasoil/Diesel,2026,1,2026-01-01,416.0
1,Gasoil/Diesel,2026,2,2026-02-01,439.0
2,Gasoil/Diesel,2026,3,2026-03-01,770.0
3,Gasoil/Diesel,2026,4,2026-04-01,140.0
4,Gasoil/Diesel,2026,5,2026-05-01,360.0
5,Gasoil/Diesel,2026,6,2026-06-01,323.0
6,Gasoil/Diesel,2026,7,2026-07-01,289.0
7,Gasoil/Diesel,2026,8,2026-08-01,0.0
8,Gasoil/Diesel,2026,9,2026-09-01,0.0
9,Gasoil/Diesel,2026,10,2026-10-01,0.0


In [35]:
quotas_wide = build_export_monthly_from_quotas(quotas_monthly)

print(f"quotas_wide  shape={quotas_wide.shape}")
print("Columns:", list(quotas_wide.columns))
display(quotas_wide.head(36))
display(quotas_wide.tail(6))

quotas_wide  shape=(12, 5)
Columns: ['month', 'Gasoline', 'Jet', 'Gasoil/Diesel', 'FO']


product,month,Gasoline,Jet,Gasoil/Diesel,FO
0,2026-01,483.0,2431.5,416.0,0.0
1,2026-02,272.0,2159.0,439.0,0.0
2,2026-03,556.0,2550.0,770.0,0.0
3,2026-04,10.0,1186.2,140.0,0.0
4,2026-05,170.0,470.0,360.0,0.0
5,2026-06,51.0,1364.5,323.0,0.0
6,2026-07,83.5,1531.3,289.0,0.0
7,2026-08,0.0,0.0,0.0,0.0
8,2026-09,0.0,0.0,0.0,0.0
9,2026-10,0.0,0.0,0.0,0.0


product,month,Gasoline,Jet,Gasoil/Diesel,FO
6,2026-07,83.5,1531.3,289.0,0.0
7,2026-08,0.0,0.0,0.0,0.0
8,2026-09,0.0,0.0,0.0,0.0
9,2026-10,0.0,0.0,0.0,0.0
10,2026-11,0.0,0.0,0.0,0.0
11,2026-12,0.0,0.0,0.0,0.0


## 11. Quick sanity checks

Re-run after a cache refresh to validate row counts and date spans.

In [31]:
checks = []

w = frames["weekly"]
checks.append(("weekly date span", w["week_start"].min(), w["week_start"].max(), len(w)))

c = frames["consumption_national"]
checks.append(("consumption years", c["year"].min(), c["year"].max(), len(c)))

j = frames["kerosene_refueling_national"]
checks.append(("jet fuel rows", j["year"].min(), j["year"].max(), len(j)))

pd.DataFrame(checks, columns=["dataset", "min", "max", "rows"])

,dataset,min,max,rows
0,weekly date span,2021-01-08 00:00:00,2026-07-03 00:00:00,3894
1,consumption years,2021,2026,396
2,jet fuel rows,2021,2026,198


In [32]:
frames["consumption_national"].to_excel('cosumption.xlsx')

In [43]:
[x for x in range(101,703)]

[101,
 102,
 103,
 104,
 105,
 106,
 107,
 108,
 109,
 110,
 111,
 112,
 113,
 114,
 115,
 116,
 117,
 118,
 119,
 120,
 121,
 122,
 123,
 124,
 125,
 126,
 127,
 128,
 129,
 130,
 131,
 132,
 133,
 134,
 135,
 136,
 137,
 138,
 139,
 140,
 141,
 142,
 143,
 144,
 145,
 146,
 147,
 148,
 149,
 150,
 151,
 152,
 153,
 154,
 155,
 156,
 157,
 158,
 159,
 160,
 161,
 162,
 163,
 164,
 165,
 166,
 167,
 168,
 169,
 170,
 171,
 172,
 173,
 174,
 175,
 176,
 177,
 178,
 179,
 180,
 181,
 182,
 183,
 184,
 185,
 186,
 187,
 188,
 189,
 190,
 191,
 192,
 193,
 194,
 195,
 196,
 197,
 198,
 199,
 200,
 201,
 202,
 203,
 204,
 205,
 206,
 207,
 208,
 209,
 210,
 211,
 212,
 213,
 214,
 215,
 216,
 217,
 218,
 219,
 220,
 221,
 222,
 223,
 224,
 225,
 226,
 227,
 228,
 229,
 230,
 231,
 232,
 233,
 234,
 235,
 236,
 237,
 238,
 239,
 240,
 241,
 242,
 243,
 244,
 245,
 246,
 247,
 248,
 249,
 250,
 251,
 252,
 253,
 254,
 255,
 256,
 257,
 258,
 259,
 260,
 261,
 262,
 263,
 264,
 265,
 266,
 267